In [2]:
import ast
import os

import cv2
import numpy as np
import pandas as pd


def draw_border(img, top_left, bottom_right, color=(0, 255, 0), thickness=10, line_length_x=200, line_length_y=200):
    x1, y1 = top_left
    x2, y2 = bottom_right

    cv2.line(img, (x1, y1), (x1, y1 + line_length_y), color, thickness)  #-- top-left
    cv2.line(img, (x1, y1), (x1 + line_length_x, y1), color, thickness)

    cv2.line(img, (x1, y2), (x1, y2 - line_length_y), color, thickness)  #-- bottom-left
    cv2.line(img, (x1, y2), (x1 + line_length_x, y2), color, thickness)

    cv2.line(img, (x2, y1), (x2 - line_length_x, y1), color, thickness)  #-- top-right
    cv2.line(img, (x2, y1), (x2, y1 + line_length_y), color, thickness)

    cv2.line(img, (x2, y2), (x2, y2 - line_length_y), color, thickness)  #-- bottom-right
    cv2.line(img, (x2, y2), (x2 - line_length_x, y2), color, thickness)

    return img


def parse_bbox(value):
    """Parse a '[x1 y1 x2 y2]' bbox string (whitespace separated) into 4 floats."""
    if isinstance(value, str):
        return [float(v) for v in value.strip('[]').split()]
    return list(value)


CSV_PATH = './Output/detections_interpolated.csv'
VIDEO_PATH = './Input/2103099-uhd_3840_2160_30fps.mp4'
OUTPUT_PATH = './Output/out.mp4'

results = pd.read_csv(CSV_PATH)
if results.empty:
    raise ValueError(
        f'{CSV_PATH} has no rows. Run OCR.ipynb and then utils.ipynb first.'
    )

# load video
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f'Could not open video: {VIDEO_PATH}')

fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Specify the codec
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))
if not out.isOpened():
    raise RuntimeError(f'Could not open VideoWriter for {OUTPUT_PATH}')

license_plate = {}
for car_id in np.unique(results['car_id']):
    car_rows = results[results['car_id'] == car_id]
    max_ = np.amax(car_rows['license_number_score'])
    best = car_rows[car_rows['license_number_score'] == max_].iloc[0]

    cap.set(cv2.CAP_PROP_POS_FRAMES, int(best['frame_nmr']))
    ret, frame = cap.read()
    if not ret:
        print(f'skipping car {car_id}: could not read frame {int(best["frame_nmr"])}')
        continue

    x1, y1, x2, y2 = parse_bbox(best['license_plate_bbox'])

    # clamp to the frame so the crop is never empty
    x1, x2 = max(0, int(x1)), min(width, int(x2))
    y1, y2 = max(0, int(y1)), min(height, int(y2))
    if x2 <= x1 or y2 <= y1:
        print(f'skipping car {car_id}: degenerate license plate bbox')
        continue

    license_crop = frame[y1:y2, x1:x2, :]
    license_crop = cv2.resize(license_crop, (max(1, int((x2 - x1) * 400 / (y2 - y1))), 400))

    license_plate[car_id] = {
        'license_crop': license_crop,
        'license_plate_number': str(best['license_number']),
    }

print(f'built crops for {len(license_plate)} cars')

frame_nmr = -1

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# read frames
ret = True
while ret:
    ret, frame = cap.read()
    frame_nmr += 1
    if ret:
        df_ = results[results['frame_nmr'] == frame_nmr]
        for row_indx in range(len(df_)):
            row = df_.iloc[row_indx]
            if row['car_id'] not in license_plate:
                continue

            # draw car
            car_x1, car_y1, car_x2, car_y2 = parse_bbox(row['car_bbox'])
            draw_border(frame, (int(car_x1), int(car_y1)), (int(car_x2), int(car_y2)), (0, 255, 0), 25,
                        line_length_x=200, line_length_y=200)

            # draw license plate
            x1, y1, x2, y2 = parse_bbox(row['license_plate_bbox'])
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 12)

            # crop license plate
            license_crop = license_plate[row['car_id']]['license_crop']

            H, W, _ = license_crop.shape

            # the overlay only fits when the car has enough headroom in the frame
            top = int(car_y1) - H - 400
            left = int((car_x2 + car_x1 - W) / 2)
            right = left + W
            if top < 0 or left < 0 or right > width:
                continue

            frame[int(car_y1) - H - 100:int(car_y1) - 100, left:right, :] = license_crop
            frame[top:int(car_y1) - H - 100, left:right, :] = (255, 255, 255)

            text = license_plate[row['car_id']]['license_plate_number']
            (text_width, text_height), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 4.3, 17)

            cv2.putText(frame,
                        text,
                        (int((car_x2 + car_x1 - text_width) / 2), int(car_y1 - H - 250 + (text_height / 2))),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        4.3,
                        (0, 0, 0),
                        17)

        out.write(frame)

out.release()
cap.release()
print(f'wrote {OUTPUT_PATH} ({frame_nmr} frames)')


built crops for 37 cars
wrote ./Output/out.mp4 (1800 frames)
